# CFM vs standard MLP: training checkpoints and evaluation-step sweep

Train **100 networks per method at each width: 32, 64, and 128**, for **32,000 optimizer updates per network** (600 independent training runs). Save the same model trajectory at **1,000, 2,000, 4,000, 8,000, 16,000, and 32,000 steps**. These are six snapshots of each run, not separately initialized networks.

Methods share the update count, learning rate, training points, evaluation grid, and seed IDs. CFM has two inputs and `4*hidden_size+1` parameters; the standard MLP has one input and `3*hidden_size+1` parameters. Paired seeds do not imply identical initial parameters.

Each `.pt` checkpoint contains a CPU `state_dict`, method, prediction type, training configuration, seed, actual completed step count, and elapsed training time. A completed network is skipped only when **all six compatible checkpoints exist**. An interrupted network with missing snapshots restarts from its seed; optimizer state is not restored.

**Evaluation reloads each checkpoint into a fresh model.** At every saved training step, evaluate the standard MLP directly and CFM at **1, 2, 4, 8, 16, 32, and 64 Euler steps**, starting from zero noise for every x. Changing the evaluation-step list or `n_grid` does not require retraining. A full batch produces 3,600 weight files and 14,400 metric rows.

The statistics table includes every training checkpoint, hidden width, and evaluation variant. The box plot has one panel per width, individual MSE values, and highlighted **arithmetic means**; its dropdown selects the training checkpoint. All six training stages remain available without recomputing metrics. Means use the original MSE values; the plot axis is logarithmic.

Weights are stored under `notebooks/outputs/cfm_vs_standard_checkpoints/<method>/width_<m>/seed_<s>/step_<n>.pt`. Recomputed metrics go to `notebooks/outputs/cfm_vs_standard_metrics.csv` with separate `training_steps` and `eval_sampling_steps` columns. Both evaluation and reporting support partial batches.

In [1]:
from pathlib import Path
import sys
import csv
import json
from dataclasses import asdict, replace
from time import perf_counter

project_root = next(
    p
    for p in (Path.cwd(), *Path.cwd().parents)
    if (p / "pyproject.toml").is_file() and (p / "notebooks" / "_shared").is_dir()
)
shared_parent = str(project_root / "notebooks")
if shared_parent not in sys.path:
    sys.path.insert(0, shared_parent)

import numpy as np
import torch
import ipywidgets as widgets
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from IPython.display import display, Markdown
from _shared import Config, Experiment
from _shared.conditional_flow_matching import FlowMatchingConfig
from _shared.flow_matching_experiment import FlowMatchingExperiment
from _shared.comparison import (
    METHODS,
    METRIC_COLUMNS,
    checkpoint_path,
    save_checkpoint,
    load_checkpoint,
    evaluate_checkpoint,
    variant_label,
)

torch.set_num_threads(1)

In [2]:
n_runs = 100  # Per method and width: 600 independent training runs.
hidden_sizes = (32, 64, 128)
checkpoint_steps = (1_000, 2_000, 4_000, 8_000, 16_000, 32_000)
cfm_eval_sampling_steps = (1, 2, 4, 8, 16, 32, 64)
config = Config(
    total_steps=32_000,
    learning_rate=0.01,
    seed=42,
    n_train=256,
    n_grid=601,
    x_min=-3.0,
    x_max=3.0,
)
flow_config = FlowMatchingConfig(
    eval_sampling_steps=1,  # Overridden independently for each evaluation variant.
    train_sampling_steps=None,
)
methods = METHODS
output_dir = project_root / "notebooks" / "outputs"
checkpoint_dir = output_dir / "cfm_vs_standard_checkpoints"
metrics_path = output_dir / "cfm_vs_standard_metrics.csv"
metadata_path = metrics_path.with_name(metrics_path.stem + "_config.json")

if type(n_runs) is not int or n_runs < 1:
    raise ValueError("n_runs must be a positive integer.")
if not hidden_sizes or any(
    type(width) is not int or width < 1 for width in hidden_sizes
):
    raise ValueError("hidden_sizes must contain positive integers.")
if len(set(hidden_sizes)) != len(hidden_sizes):
    raise ValueError("hidden_sizes must not contain duplicates.")
if not checkpoint_steps or any(
    type(step) is not int or not 1 <= step <= config.total_steps
    for step in checkpoint_steps
):
    raise ValueError(
        "checkpoint_steps must contain integers between 1 and total_steps."
    )
if tuple(sorted(set(checkpoint_steps))) != tuple(checkpoint_steps):
    raise ValueError("checkpoint_steps must be strictly increasing.")
if checkpoint_steps[-1] != config.total_steps:
    raise ValueError("checkpoint_steps must include the final training step.")
if not cfm_eval_sampling_steps or any(
    type(step) is not int or step < 1 for step in cfm_eval_sampling_steps
):
    raise ValueError("cfm_eval_sampling_steps must contain positive integers.")
if len(set(cfm_eval_sampling_steps)) != len(cfm_eval_sampling_steps):
    raise ValueError("cfm_eval_sampling_steps must not contain duplicates.")

tasks = [
    (width, method, config.seed + run_index)
    for width in hidden_sizes
    for run_index in range(n_runs)
    for method in methods
]
evaluation_tasks = [(*task, step) for task in tasks for step in checkpoint_steps]
variants = ["Standard MLP"] + [
    variant_label("CFM", steps) for steps in cfm_eval_sampling_steps
]

In [3]:
completed = widgets.IntProgress(
    value=0,
    min=0,
    max=len(tasks),
    description="Networks:",
    layout=widgets.Layout(width="100%"),
)
current = widgets.HTML()
display(widgets.VBox([completed, current]))
progress_every = max(1, config.total_steps // 100)
checkpoint_step_set = set(checkpoint_steps)
trained_count = 0
reused_count = 0

for width, method, seed in tasks:
    run_config = replace(config, hidden_size=width, seed=seed)
    paths = {
        step: checkpoint_path(checkpoint_dir, method, width, seed, step)
        for step in checkpoint_steps
    }
    for step, path in paths.items():
        if path.is_file():
            payload = load_checkpoint(
                path,
                method=method,
                config=run_config,
                flow_config=flow_config,
                training_steps=step,
            )
            del payload
    if all(path.is_file() for path in paths.values()):
        reused_count += 1
        completed.value += 1
        current.value = f"Reused all {len(paths)} checkpoints: <b>{method}</b> | {width} neurons | seed {seed}"
        continue
    # Restart incomplete trajectories from their seed to recover every snapshot.
    experiment = (
        Experiment(run_config)
        if method == "Standard MLP"
        else FlowMatchingExperiment(run_config, flow_config)
    )
    started = perf_counter()
    for progress in experiment.run():
        if progress.step in checkpoint_step_set:
            save_checkpoint(
                paths[progress.step], experiment, method, perf_counter() - started
            )
        if progress.step % progress_every == 0 or progress.step in checkpoint_step_set:
            current.value = (
                f"<b>{method}</b> | {width} neurons | run {seed - config.seed + 1}/{n_runs}"
                f" | seed {seed} | step {progress.step:,}/{config.total_steps:,}"
            )
    del experiment
    trained_count += 1
    completed.value += 1

completed.bar_style = "success"
current.value = (
    f"Ready: {completed.value}/{len(tasks)} networks, {len(evaluation_tasks)} checkpoints. "
    f"Trained {trained_count}, reused {reused_count}. Run evaluation to compute metrics at every checkpoint."
)

## Recompute metrics at every training checkpoint

This cell performs **no training**. In a fresh kernel, run imports and configuration, then run evaluation. It reloads every available `(width, method, seed, training_steps)` checkpoint. Each CFM snapshot is evaluated with every value in `cfm_eval_sampling_steps`; the standard MLP is evaluated directly. Missing checkpoints are reported and skipped for partial-batch inspection.

Changing `cfm_eval_sampling_steps` or `n_grid` only requires rerunning configuration, evaluation, statistics, and plots. Metric rows distinguish optimizer updates (`training_steps`) from Euler evaluation steps (`eval_sampling_steps`). A standard MLP row uses `eval_sampling_steps=0` for direct prediction.

In [4]:
results = []
missing_checkpoints = []
output_dir.mkdir(parents=True, exist_ok=True)
metadata_path.write_text(
    json.dumps(
        {
            "hidden_sizes": list(hidden_sizes),
            "n_runs_per_method_per_size": n_runs,
            "checkpoint_steps": list(checkpoint_steps),
            "config": asdict(config),
            "flow_config": asdict(flow_config),
            "cfm_eval_sampling_steps": list(cfm_eval_sampling_steps),
            "initial_noise": 0,
            "checkpoint_dir": str(checkpoint_dir),
            "primary_metric": "grid_mse",
        },
        indent=2,
    )
    + "\n",
    encoding="utf-8",
)
with metrics_path.open("w", newline="", encoding="utf-8") as stream:
    csv.DictWriter(stream, fieldnames=METRIC_COLUMNS).writeheader()

evaluated = widgets.IntProgress(
    value=0,
    min=0,
    max=len(evaluation_tasks),
    description="Evaluation:",
    layout=widgets.Layout(width="100%"),
)
evaluation_status = widgets.HTML()
display(widgets.VBox([evaluated, evaluation_status]))
for width, method, seed, training_step in evaluation_tasks:
    path = checkpoint_path(checkpoint_dir, method, width, seed, training_step)
    if not path.is_file():
        missing_checkpoints.append(str(path))
        evaluated.value += 1
        continue
    evaluation_status.value = (
        f"<b>{method}</b> | {width} neurons | seed {seed}"
        f" | {training_step:,} training steps | evaluating saved weights"
    )
    payload = load_checkpoint(
        path,
        method=method,
        config=replace(config, hidden_size=width, seed=seed),
        flow_config=flow_config,
        training_steps=training_step,
    )
    rows = evaluate_checkpoint(
        payload, path, eval_sampling_steps=cfm_eval_sampling_steps, n_grid=config.n_grid
    )
    with metrics_path.open("a", newline="", encoding="utf-8") as stream:
        csv.DictWriter(stream, fieldnames=METRIC_COLUMNS).writerows(rows)
    results.extend(rows)
    evaluated.value += 1
    del payload

evaluated.bar_style = "warning" if missing_checkpoints else "success"
evaluation_status.value = (
    f"Computed {len(results)} metric rows from {len(evaluation_tasks) - len(missing_checkpoints)} checkpoints. "
    f"Missing checkpoints: {len(missing_checkpoints)}. Saved to <code>{metrics_path.name}</code>."
)

In [5]:
# Optional: reload computed metrics without training or evaluation.
with metrics_path.open(newline="", encoding="utf-8") as stream:
    reader = csv.DictReader(stream)
    if "training_steps" not in (reader.fieldnames or []):
        raise ValueError(
            "The metrics CSV must include a training_steps column. Run checkpoint evaluation to generate it."
        )
    results = [
        {
            "training_steps": int(row["training_steps"]),
            "hidden_size": int(row["hidden_size"]),
            "method": row["method"],
            "eval_sampling_steps": int(row["eval_sampling_steps"]),
            "variant": row["variant"],
            "seed": int(row["seed"]),
            "train_mse": float(row["train_mse"]),
            "grid_mse": float(row["grid_mse"]),
            "training_seconds": float(row["training_seconds"]),
            "checkpoint": row["checkpoint"],
        }
        for row in reader
    ]

In [6]:
if not results:
    raise ValueError(
        "No metrics are available. Evaluate saved checkpoints or load computed metrics first."
    )

summary = []
for training_step in checkpoint_steps:
    for width in hidden_sizes:
        for variant in variants:
            values = np.array(
                [
                    row["grid_mse"]
                    for row in results
                    if row["training_steps"] == training_step
                    and row["hidden_size"] == width
                    and row["variant"] == variant
                    and config.seed <= row["seed"] < config.seed + n_runs
                ]
            )
            if not len(values):
                continue
            summary.append(
                {
                    "training_steps": training_step,
                    "hidden_size": width,
                    "variant": variant,
                    "runs": len(values),
                    "mean": float(values.mean()),
                    "std": (
                        float(values.std(ddof=1)) if len(values) > 1 else float("nan")
                    ),
                    "min": float(values.min()),
                    "q25": float(np.quantile(values, 0.25)),
                    "median": float(np.median(values)),
                    "q75": float(np.quantile(values, 0.75)),
                    "max": float(values.max()),
                }
            )
if not summary:
    raise ValueError(
        "No metrics match the requested training checkpoints, widths, variants, and seeds."
    )

for training_step in checkpoint_steps:
    stage_rows = [row for row in summary if row["training_steps"] == training_step]
    if not stage_rows:
        continue
    table = [
        "| Hidden neurons | Model / Euler steps | Runs | Mean | Sample std | Min | Q25 | Median | Q75 | Max |",
        "|--:|:--|--:|--:|--:|--:|--:|--:|--:|--:|",
    ]
    for row in stage_rows:
        numbers = [
            f"{row[key]:.6g}"
            for key in ("mean", "std", "min", "q25", "median", "q75", "max")
        ]
        table.append(
            f"| {row['hidden_size']} | {row['variant']} | {row['runs']} | "
            + " | ".join(numbers)
            + " |"
        )
    display(
        Markdown(
            f"### Grid MSE at {training_step:,} training steps\n\n" + "\n".join(table)
        )
    )

### Grid MSE at 1,000 training steps

| Hidden neurons | Model / Euler steps | Runs | Mean | Sample std | Min | Q25 | Median | Q75 | Max |
|--:|:--|--:|--:|--:|--:|--:|--:|--:|--:|
| 32 | Standard MLP | 100 | 0.042068 | 0.0193944 | 0.00761107 | 0.0290863 | 0.038941 | 0.0674819 | 0.0735118 |
| 32 | CFM 1 | 100 | 0.0167101 | 0.0133852 | 0.00690643 | 0.00923806 | 0.0105922 | 0.0171332 | 0.0748512 |
| 32 | CFM 2 | 100 | 0.0146791 | 0.0116819 | 0.00682364 | 0.00870738 | 0.00947549 | 0.0135108 | 0.0633578 |
| 32 | CFM 4 | 100 | 0.0139372 | 0.0108548 | 0.00655602 | 0.00850822 | 0.00928195 | 0.0124422 | 0.0610144 |
| 32 | CFM 8 | 100 | 0.0137833 | 0.0105891 | 0.00651532 | 0.00851932 | 0.00931804 | 0.0122718 | 0.0618939 |
| 32 | CFM 16 | 100 | 0.0138575 | 0.0106097 | 0.00663074 | 0.00864003 | 0.00953173 | 0.0126271 | 0.0634571 |
| 32 | CFM 32 | 100 | 0.0139859 | 0.0106939 | 0.00673244 | 0.00878538 | 0.00965601 | 0.0128259 | 0.0650737 |
| 32 | CFM 64 | 100 | 0.0140918 | 0.0107884 | 0.00679776 | 0.00885421 | 0.00974112 | 0.0129189 | 0.0666928 |
| 64 | Standard MLP | 100 | 0.0309087 | 0.015454 | 0.00791003 | 0.0179275 | 0.0310791 | 0.0417162 | 0.0677624 |
| 64 | CFM 1 | 100 | 0.00868596 | 0.00214618 | 0.00547284 | 0.00755596 | 0.0083068 | 0.00909973 | 0.0230023 |
| 64 | CFM 2 | 100 | 0.00795691 | 0.00168764 | 0.00521646 | 0.00711756 | 0.00781575 | 0.00842243 | 0.0204378 |
| 64 | CFM 4 | 100 | 0.00769405 | 0.00154835 | 0.00508075 | 0.00688545 | 0.00758737 | 0.00818906 | 0.0193431 |
| 64 | CFM 8 | 100 | 0.0076978 | 0.00158369 | 0.0050654 | 0.00672369 | 0.00754212 | 0.00829128 | 0.0191211 |
| 64 | CFM 16 | 100 | 0.00784609 | 0.0016095 | 0.00502689 | 0.00689542 | 0.00780266 | 0.00837733 | 0.0190876 |
| 64 | CFM 32 | 100 | 0.00799762 | 0.00164399 | 0.00502095 | 0.00714374 | 0.00792651 | 0.0086287 | 0.019079 |
| 64 | CFM 64 | 100 | 0.00810556 | 0.00167783 | 0.00502653 | 0.00717666 | 0.00798245 | 0.00866932 | 0.0190731 |
| 128 | Standard MLP | 100 | 0.0180151 | 0.0100337 | 0.00734604 | 0.011467 | 0.014384 | 0.0203541 | 0.0642038 |
| 128 | CFM 1 | 100 | 0.00724099 | 0.00129635 | 0.00493549 | 0.00638646 | 0.00701368 | 0.00783671 | 0.012035 |
| 128 | CFM 2 | 100 | 0.00673782 | 0.00127522 | 0.00471666 | 0.00591858 | 0.00644961 | 0.00740636 | 0.011204 |
| 128 | CFM 4 | 100 | 0.0064942 | 0.00133117 | 0.00429285 | 0.00563674 | 0.00619123 | 0.00722168 | 0.0108812 |
| 128 | CFM 8 | 100 | 0.00648826 | 0.00138524 | 0.00403877 | 0.00553503 | 0.00617696 | 0.00708265 | 0.0110746 |
| 128 | CFM 16 | 100 | 0.00663139 | 0.00144591 | 0.00406915 | 0.00568307 | 0.00629515 | 0.00719674 | 0.0118095 |
| 128 | CFM 32 | 100 | 0.00676567 | 0.00151057 | 0.00411866 | 0.00576491 | 0.00651808 | 0.0073119 | 0.0123236 |
| 128 | CFM 64 | 100 | 0.00685536 | 0.0015586 | 0.00415101 | 0.00580012 | 0.00663364 | 0.00743382 | 0.0126365 |

### Grid MSE at 2,000 training steps

| Hidden neurons | Model / Euler steps | Runs | Mean | Sample std | Min | Q25 | Median | Q75 | Max |
|--:|:--|--:|--:|--:|--:|--:|--:|--:|--:|
| 32 | Standard MLP | 100 | 0.0343607 | 0.0228138 | 0.00330002 | 0.00921144 | 0.0321686 | 0.066734 | 0.0733426 |
| 32 | CFM 1 | 100 | 0.00759657 | 0.00490444 | 0.00211436 | 0.00562696 | 0.0070188 | 0.00833188 | 0.0474241 |
| 32 | CFM 2 | 100 | 0.00729194 | 0.0044665 | 0.00214796 | 0.0056401 | 0.00684567 | 0.00780967 | 0.0438339 |
| 32 | CFM 4 | 100 | 0.00717488 | 0.00457209 | 0.0023675 | 0.00559118 | 0.0067013 | 0.00779086 | 0.0454353 |
| 32 | CFM 8 | 100 | 0.00715986 | 0.00472812 | 0.00261151 | 0.00539931 | 0.00669657 | 0.00768019 | 0.0471888 |
| 32 | CFM 16 | 100 | 0.00725186 | 0.00488756 | 0.00266959 | 0.00547496 | 0.00678587 | 0.00775577 | 0.0489561 |
| 32 | CFM 32 | 100 | 0.00736086 | 0.00500273 | 0.00268523 | 0.00563039 | 0.0068221 | 0.00786118 | 0.0502762 |
| 32 | CFM 64 | 100 | 0.00744529 | 0.00507831 | 0.00269661 | 0.00566255 | 0.00686055 | 0.00803946 | 0.0511404 |
| 64 | Standard MLP | 100 | 0.0198661 | 0.0173402 | 0.003993 | 0.00594717 | 0.00933501 | 0.0316877 | 0.069938 |
| 64 | CFM 1 | 100 | 0.00402771 | 0.00135224 | 0.00147594 | 0.0031255 | 0.00395113 | 0.00494242 | 0.00714389 |
| 64 | CFM 2 | 100 | 0.00399224 | 0.00131705 | 0.00145617 | 0.00302843 | 0.00398302 | 0.00491061 | 0.00697317 |
| 64 | CFM 4 | 100 | 0.00395799 | 0.00135785 | 0.00143873 | 0.00303118 | 0.00385402 | 0.00486224 | 0.0074357 |
| 64 | CFM 8 | 100 | 0.0039861 | 0.00143999 | 0.00142808 | 0.00300009 | 0.00382597 | 0.00479139 | 0.00835449 |
| 64 | CFM 16 | 100 | 0.00412372 | 0.0015281 | 0.00139168 | 0.00316022 | 0.00389432 | 0.00495773 | 0.00932624 |
| 64 | CFM 32 | 100 | 0.00424912 | 0.00160388 | 0.00136919 | 0.00324817 | 0.00411535 | 0.00505519 | 0.00995909 |
| 64 | CFM 64 | 100 | 0.00433242 | 0.00165856 | 0.00135725 | 0.003308 | 0.0041627 | 0.0051675 | 0.0103363 |
| 128 | Standard MLP | 100 | 0.00909618 | 0.0101195 | 0.0029431 | 0.0040193 | 0.00475533 | 0.00671378 | 0.0474972 |
| 128 | CFM 1 | 100 | 0.00301666 | 0.0013746 | 0.00091203 | 0.0020188 | 0.00274085 | 0.00372285 | 0.0079614 |
| 128 | CFM 2 | 100 | 0.00305087 | 0.00148134 | 0.00083344 | 0.0019651 | 0.00276197 | 0.00371073 | 0.0080586 |
| 128 | CFM 4 | 100 | 0.00305676 | 0.00164106 | 0.000759977 | 0.00200268 | 0.00273683 | 0.00363724 | 0.00940401 |
| 128 | CFM 8 | 100 | 0.00309853 | 0.00172557 | 0.00074148 | 0.00194005 | 0.00275183 | 0.00365865 | 0.00953801 |
| 128 | CFM 16 | 100 | 0.00321063 | 0.00176927 | 0.000774124 | 0.00204095 | 0.00283763 | 0.00391242 | 0.00971154 |
| 128 | CFM 32 | 100 | 0.00331916 | 0.00181234 | 0.000779977 | 0.00212283 | 0.00298572 | 0.00400757 | 0.00987664 |
| 128 | CFM 64 | 100 | 0.00339334 | 0.0018495 | 0.000790163 | 0.00216392 | 0.00304838 | 0.00405198 | 0.00999605 |

### Grid MSE at 4,000 training steps

| Hidden neurons | Model / Euler steps | Runs | Mean | Sample std | Min | Q25 | Median | Q75 | Max |
|--:|:--|--:|--:|--:|--:|--:|--:|--:|--:|
| 32 | Standard MLP | 100 | 0.0323508 | 0.0235329 | 0.00215027 | 0.00644161 | 0.0318333 | 0.0478092 | 0.0680914 |
| 32 | CFM 1 | 100 | 0.00426914 | 0.00189356 | 0.00106614 | 0.0030171 | 0.00383191 | 0.00551245 | 0.00906321 |
| 32 | CFM 2 | 100 | 0.00421372 | 0.00185147 | 0.00107466 | 0.00299079 | 0.00372387 | 0.00558152 | 0.009057 |
| 32 | CFM 4 | 100 | 0.0041938 | 0.00183962 | 0.00106601 | 0.00300777 | 0.00377305 | 0.00541531 | 0.00886998 |
| 32 | CFM 8 | 100 | 0.00418945 | 0.00182473 | 0.00105612 | 0.00295888 | 0.00386846 | 0.00547251 | 0.0089803 |
| 32 | CFM 16 | 100 | 0.00427989 | 0.00186579 | 0.00105105 | 0.00297027 | 0.00392428 | 0.00568495 | 0.00920491 |
| 32 | CFM 32 | 100 | 0.00441557 | 0.00195252 | 0.00104854 | 0.0030255 | 0.00402962 | 0.0058154 | 0.00932399 |
| 32 | CFM 64 | 100 | 0.00451054 | 0.0020211 | 0.00104731 | 0.00306474 | 0.00408166 | 0.00585409 | 0.00937558 |
| 64 | Standard MLP | 100 | 0.0167966 | 0.0180244 | 0.00133649 | 0.0022595 | 0.00580124 | 0.0287964 | 0.0668846 |
| 64 | CFM 1 | 100 | 0.00207492 | 0.00125904 | 0.000199082 | 0.00119538 | 0.00176149 | 0.00284728 | 0.0070529 |
| 64 | CFM 2 | 100 | 0.00211579 | 0.00131277 | 0.000206009 | 0.00121857 | 0.00177375 | 0.00287433 | 0.006781 |
| 64 | CFM 4 | 100 | 0.00212014 | 0.0013642 | 0.000225843 | 0.00121881 | 0.00186989 | 0.0028015 | 0.00810221 |
| 64 | CFM 8 | 100 | 0.00214057 | 0.00141963 | 0.000233033 | 0.00123738 | 0.00186042 | 0.00283989 | 0.00864298 |
| 64 | CFM 16 | 100 | 0.0022258 | 0.00152974 | 0.000239451 | 0.00126855 | 0.00185114 | 0.0028878 | 0.00877538 |
| 64 | CFM 32 | 100 | 0.00230003 | 0.00162074 | 0.000243096 | 0.00130375 | 0.00187492 | 0.003032 | 0.00871197 |
| 64 | CFM 64 | 100 | 0.00234945 | 0.00167864 | 0.000244916 | 0.00131646 | 0.00187628 | 0.00311862 | 0.00859435 |
| 128 | Standard MLP | 100 | 0.00614023 | 0.00937499 | 0.0017508 | 0.00209522 | 0.00228504 | 0.00298994 | 0.0413562 |
| 128 | CFM 1 | 100 | 0.00138014 | 0.001331 | 0.00013394 | 0.000389263 | 0.00119823 | 0.0018285 | 0.00996828 |
| 128 | CFM 2 | 100 | 0.00148069 | 0.0016353 | 0.000127067 | 0.000432457 | 0.00119968 | 0.00194709 | 0.0137702 |
| 128 | CFM 4 | 100 | 0.00154591 | 0.00190585 | 0.000120951 | 0.000431962 | 0.0012026 | 0.0020416 | 0.0168128 |
| 128 | CFM 8 | 100 | 0.0015932 | 0.00203957 | 0.00011719 | 0.000420407 | 0.00122474 | 0.00209954 | 0.0181873 |
| 128 | CFM 16 | 100 | 0.00163919 | 0.00209527 | 0.000118893 | 0.000419842 | 0.00124971 | 0.00215682 | 0.0185779 |
| 128 | CFM 32 | 100 | 0.00167007 | 0.00211308 | 0.00012063 | 0.00043299 | 0.00127322 | 0.00219347 | 0.018585 |
| 128 | CFM 64 | 100 | 0.00168946 | 0.00212631 | 0.000121604 | 0.000430518 | 0.00127537 | 0.00220874 | 0.0186014 |

### Grid MSE at 8,000 training steps

| Hidden neurons | Model / Euler steps | Runs | Mean | Sample std | Min | Q25 | Median | Q75 | Max |
|--:|:--|--:|--:|--:|--:|--:|--:|--:|--:|
| 32 | Standard MLP | 100 | 0.0267672 | 0.0216579 | 0.000889881 | 0.00529678 | 0.0286673 | 0.0325902 | 0.0682102 |
| 32 | CFM 1 | 100 | 0.00284257 | 0.00180702 | 0.000393336 | 0.00159924 | 0.00217169 | 0.00376283 | 0.0087315 |
| 32 | CFM 2 | 100 | 0.00283326 | 0.00175257 | 0.00039507 | 0.00163841 | 0.00223268 | 0.0037216 | 0.00859789 |
| 32 | CFM 4 | 100 | 0.00283407 | 0.00175667 | 0.000370388 | 0.00160879 | 0.00235553 | 0.00363588 | 0.008502 |
| 32 | CFM 8 | 100 | 0.00283742 | 0.00172766 | 0.000374501 | 0.00164913 | 0.0023609 | 0.00352814 | 0.0082457 |
| 32 | CFM 16 | 100 | 0.00286684 | 0.00171615 | 0.000377067 | 0.00169301 | 0.00238191 | 0.00353337 | 0.00776444 |
| 32 | CFM 32 | 100 | 0.00293131 | 0.0017648 | 0.000380933 | 0.00170157 | 0.00244451 | 0.00358875 | 0.00839691 |
| 32 | CFM 64 | 100 | 0.00298137 | 0.00181708 | 0.000381758 | 0.0017049 | 0.00246472 | 0.0037046 | 0.00914734 |
| 64 | Standard MLP | 100 | 0.0126591 | 0.0162211 | 0.000344978 | 0.00185254 | 0.00270967 | 0.0270745 | 0.0664824 |
| 64 | CFM 1 | 100 | 0.00126538 | 0.000941981 | 0.000187283 | 0.000564898 | 0.00117603 | 0.00162651 | 0.0060735 |
| 64 | CFM 2 | 100 | 0.00132535 | 0.000980387 | 0.000204886 | 0.00062156 | 0.00117784 | 0.00172637 | 0.00621893 |
| 64 | CFM 4 | 100 | 0.00136554 | 0.00103294 | 0.000216616 | 0.000650862 | 0.00113154 | 0.00171306 | 0.00626075 |
| 64 | CFM 8 | 100 | 0.00140902 | 0.00108852 | 0.00021376 | 0.000687461 | 0.00113517 | 0.00173395 | 0.00621035 |
| 64 | CFM 16 | 100 | 0.00145409 | 0.00114365 | 0.000211432 | 0.000695875 | 0.00116657 | 0.00174923 | 0.0062207 |
| 64 | CFM 32 | 100 | 0.0014941 | 0.00118716 | 0.000210541 | 0.000712055 | 0.00119422 | 0.00181656 | 0.00623056 |
| 64 | CFM 64 | 100 | 0.00152122 | 0.001216 | 0.000210077 | 0.00072119 | 0.00122057 | 0.00186977 | 0.00625179 |
| 128 | Standard MLP | 100 | 0.00356137 | 0.00603475 | 0.000519896 | 0.00115331 | 0.00154398 | 0.00218153 | 0.0275375 |
| 128 | CFM 1 | 100 | 0.00082831 | 0.00100296 | 8.24861e-05 | 0.000244973 | 0.00042879 | 0.00102973 | 0.00593614 |
| 128 | CFM 2 | 100 | 0.00090987 | 0.0010875 | 9.46542e-05 | 0.000249738 | 0.000451328 | 0.00115501 | 0.00552303 |
| 128 | CFM 4 | 100 | 0.000983157 | 0.00120665 | 9.19428e-05 | 0.000255797 | 0.000490131 | 0.00118311 | 0.00653964 |
| 128 | CFM 8 | 100 | 0.00102541 | 0.0012713 | 9.13011e-05 | 0.000263347 | 0.000506185 | 0.00127714 | 0.00715152 |
| 128 | CFM 16 | 100 | 0.00104884 | 0.00130772 | 9.03834e-05 | 0.000267152 | 0.000502348 | 0.00134229 | 0.00756381 |
| 128 | CFM 32 | 100 | 0.00106098 | 0.00132703 | 9.07562e-05 | 0.000269763 | 0.000504534 | 0.00136443 | 0.00776248 |
| 128 | CFM 64 | 100 | 0.00106627 | 0.00133491 | 9.10733e-05 | 0.000271702 | 0.000505762 | 0.00137763 | 0.0078624 |

### Grid MSE at 16,000 training steps

| Hidden neurons | Model / Euler steps | Runs | Mean | Sample std | Min | Q25 | Median | Q75 | Max |
|--:|:--|--:|--:|--:|--:|--:|--:|--:|--:|
| 32 | Standard MLP | 100 | 0.0209645 | 0.0203917 | 0.000415724 | 0.00347008 | 0.0177325 | 0.0304183 | 0.0681715 |
| 32 | CFM 1 | 100 | 0.00188646 | 0.00143534 | 0.000365271 | 0.000857623 | 0.00155249 | 0.00236654 | 0.00827154 |
| 32 | CFM 2 | 100 | 0.00191211 | 0.0014538 | 0.00034893 | 0.000805285 | 0.00156853 | 0.00252116 | 0.00827927 |
| 32 | CFM 4 | 100 | 0.00193192 | 0.00147186 | 0.000350167 | 0.000802397 | 0.00159113 | 0.00253537 | 0.00830106 |
| 32 | CFM 8 | 100 | 0.00195336 | 0.00148176 | 0.000351828 | 0.000805211 | 0.0016167 | 0.0025769 | 0.00821005 |
| 32 | CFM 16 | 100 | 0.00196226 | 0.00148374 | 0.000352131 | 0.000807863 | 0.00165764 | 0.00256629 | 0.00770269 |
| 32 | CFM 32 | 100 | 0.00198293 | 0.00150587 | 0.000352079 | 0.000810221 | 0.00169556 | 0.00258854 | 0.00781066 |
| 32 | CFM 64 | 100 | 0.00199941 | 0.00152818 | 0.000352146 | 0.00081165 | 0.00170192 | 0.00258801 | 0.00807913 |
| 64 | Standard MLP | 100 | 0.00399612 | 0.00805351 | 0.000123997 | 0.000547944 | 0.00167177 | 0.00249485 | 0.0410181 |
| 64 | CFM 1 | 100 | 0.00077124 | 0.000691309 | 9.94726e-05 | 0.000347842 | 0.000523803 | 0.00101811 | 0.00432638 |
| 64 | CFM 2 | 100 | 0.000828323 | 0.000735046 | 0.000101766 | 0.000352922 | 0.000593935 | 0.00107153 | 0.00454996 |
| 64 | CFM 4 | 100 | 0.000870457 | 0.000778065 | 0.000105796 | 0.000357034 | 0.000600164 | 0.0011036 | 0.0044766 |
| 64 | CFM 8 | 100 | 0.000899232 | 0.000812828 | 0.000107393 | 0.000364873 | 0.00061215 | 0.00114327 | 0.0044523 |
| 64 | CFM 16 | 100 | 0.000913551 | 0.000828999 | 0.000108304 | 0.000366199 | 0.000608541 | 0.00116666 | 0.00443925 |
| 64 | CFM 32 | 100 | 0.00092283 | 0.00083456 | 0.000108726 | 0.000369485 | 0.00059833 | 0.00117031 | 0.00443119 |
| 64 | CFM 64 | 100 | 0.000928475 | 0.000836913 | 0.000108893 | 0.000371669 | 0.000609144 | 0.0011719 | 0.00442779 |
| 128 | Standard MLP | 100 | 0.00128952 | 0.00299461 | 6.16824e-05 | 0.000253873 | 0.000364325 | 0.000769743 | 0.0232242 |
| 128 | CFM 1 | 100 | 0.00106574 | 0.00211602 | 8.78367e-05 | 0.000158208 | 0.000320456 | 0.000971515 | 0.0154469 |
| 128 | CFM 2 | 100 | 0.00114543 | 0.00223556 | 8.85684e-05 | 0.00016339 | 0.000361954 | 0.00109694 | 0.016797 |
| 128 | CFM 4 | 100 | 0.00121041 | 0.00233475 | 9.148e-05 | 0.000166405 | 0.000379742 | 0.00123814 | 0.0176037 |
| 128 | CFM 8 | 100 | 0.0012555 | 0.00241017 | 9.23229e-05 | 0.000174932 | 0.000378291 | 0.00131932 | 0.0182448 |
| 128 | CFM 16 | 100 | 0.00127973 | 0.00244884 | 9.25635e-05 | 0.000181843 | 0.000400703 | 0.00131056 | 0.0185844 |
| 128 | CFM 32 | 100 | 0.00129307 | 0.00247019 | 9.24365e-05 | 0.00018565 | 0.000408163 | 0.00130435 | 0.0187654 |
| 128 | CFM 64 | 100 | 0.00130008 | 0.0024815 | 9.24192e-05 | 0.000187641 | 0.00040921 | 0.00130234 | 0.0188624 |

### Grid MSE at 32,000 training steps

| Hidden neurons | Model / Euler steps | Runs | Mean | Sample std | Min | Q25 | Median | Q75 | Max |
|--:|:--|--:|--:|--:|--:|--:|--:|--:|--:|
| 32 | Standard MLP | 100 | 0.0161153 | 0.0187193 | 0.000144225 | 0.00206569 | 0.00517721 | 0.0286523 | 0.0664114 |
| 32 | CFM 1 | 100 | 0.0012068 | 0.00110732 | 0.000267464 | 0.00045017 | 0.000728853 | 0.00163483 | 0.00600869 |
| 32 | CFM 2 | 100 | 0.00123711 | 0.0011883 | 0.000270251 | 0.000469669 | 0.000722271 | 0.00160766 | 0.00767401 |
| 32 | CFM 4 | 100 | 0.00126999 | 0.00128495 | 0.000266784 | 0.000479175 | 0.00074533 | 0.00159726 | 0.00914584 |
| 32 | CFM 8 | 100 | 0.00130177 | 0.00134153 | 0.000265503 | 0.000485928 | 0.000768612 | 0.00159716 | 0.00988489 |
| 32 | CFM 16 | 100 | 0.00132958 | 0.00138701 | 0.000266011 | 0.000488823 | 0.000784793 | 0.00163202 | 0.0102693 |
| 32 | CFM 32 | 100 | 0.00134595 | 0.00141397 | 0.000266439 | 0.000489995 | 0.000788724 | 0.00166378 | 0.0104676 |
| 32 | CFM 64 | 100 | 0.0013549 | 0.00142946 | 0.00026675 | 0.000490523 | 0.000790746 | 0.00167932 | 0.0105707 |
| 64 | Standard MLP | 100 | 0.00142113 | 0.00335474 | 9.70898e-05 | 0.000244616 | 0.00041818 | 0.00120019 | 0.0268174 |
| 64 | CFM 1 | 100 | 0.000494333 | 0.00059254 | 9.4575e-05 | 0.000176331 | 0.000346492 | 0.000527878 | 0.00448989 |
| 64 | CFM 2 | 100 | 0.00053992 | 0.000694867 | 9.25379e-05 | 0.000182002 | 0.000354678 | 0.000564315 | 0.00554995 |
| 64 | CFM 4 | 100 | 0.000580428 | 0.000808996 | 9.37391e-05 | 0.000191219 | 0.000363964 | 0.00061971 | 0.00668547 |
| 64 | CFM 8 | 100 | 0.000603964 | 0.000873885 | 9.51928e-05 | 0.000194586 | 0.000369033 | 0.000661185 | 0.00732604 |
| 64 | CFM 16 | 100 | 0.000618087 | 0.000912451 | 9.59648e-05 | 0.000195587 | 0.000376692 | 0.000664383 | 0.00771151 |
| 64 | CFM 32 | 100 | 0.000625848 | 0.0009334 | 9.66339e-05 | 0.000196399 | 0.000381462 | 0.0006705 | 0.00792543 |
| 64 | CFM 64 | 100 | 0.000630111 | 0.000944595 | 9.70141e-05 | 0.000196924 | 0.000383679 | 0.000676135 | 0.00803961 |
| 128 | Standard MLP | 100 | 0.00108402 | 0.00287585 | 4.5774e-05 | 0.00012287 | 0.000196832 | 0.00043666 | 0.0158957 |
| 128 | CFM 1 | 100 | 0.000844834 | 0.00136677 | 5.90342e-05 | 0.000152747 | 0.000356889 | 0.000850137 | 0.00780096 |
| 128 | CFM 2 | 100 | 0.000936739 | 0.00147383 | 5.59674e-05 | 0.000166181 | 0.000401716 | 0.000989616 | 0.00855603 |
| 128 | CFM 4 | 100 | 0.00100736 | 0.00160005 | 5.43015e-05 | 0.000172925 | 0.000412611 | 0.00109571 | 0.00940462 |
| 128 | CFM 8 | 100 | 0.00104806 | 0.00167411 | 5.3721e-05 | 0.000179915 | 0.000419099 | 0.00111788 | 0.00990639 |
| 128 | CFM 16 | 100 | 0.00107006 | 0.00171115 | 5.35163e-05 | 0.000181781 | 0.000416914 | 0.00112459 | 0.0101365 |
| 128 | CFM 32 | 100 | 0.00108167 | 0.00173153 | 5.34957e-05 | 0.000183301 | 0.000415963 | 0.00112845 | 0.0102769 |
| 128 | CFM 64 | 100 | 0.00108768 | 0.00174237 | 5.34556e-05 | 0.000184159 | 0.000415595 | 0.00113033 | 0.0103568 |

In [7]:
report_widths = [
    width
    for width in hidden_sizes
    if any(row["hidden_size"] == width for row in summary)
]
report_steps = [
    step
    for step in checkpoint_steps
    if any(row["training_steps"] == step for row in summary)
]
selected_step = report_steps[-1]
figure = make_subplots(
    rows=len(report_widths),
    cols=1,
    shared_yaxes="all",
    subplot_titles=[f"{width} hidden neurons" for width in report_widths],
    vertical_spacing=0.12 if len(report_widths) > 1 else 0,
)
cfm_colors = (
    "#15803d",
    "#16a34a",
    "#059669",
    "#0d9488",
    "#0891b2",
    "#0284c7",
    "#0369a1",
)
colors = {"Standard MLP": "#2563eb"}
colors.update(
    {
        variant_label("CFM", step): cfm_colors[index % len(cfm_colors)]
        for index, step in enumerate(cfm_eval_sampling_steps)
    }
)
plot_floor = np.finfo(float).tiny
trace_steps = []
for training_step in report_steps:
    show_mean_legend = True
    for panel, width in enumerate(report_widths, start=1):
        for variant in variants:
            rows = [
                row
                for row in results
                if row["training_steps"] == training_step
                and row["hidden_size"] == width
                and row["variant"] == variant
                and config.seed <= row["seed"] < config.seed + n_runs
            ]
            if not rows:
                continue
            figure.add_trace(
                go.Box(
                    name=variant,
                    showlegend=False,
                    visible=training_step == selected_step,
                    x=[variant] * len(rows),
                    y=[max(row["grid_mse"], plot_floor) for row in rows],
                    customdata=[[row["seed"], row["train_mse"]] for row in rows],
                    boxpoints="all",
                    jitter=0.35,
                    pointpos=0,
                    quartilemethod="linear",
                    marker=dict(color=colors[variant], size=4, opacity=0.5),
                    line=dict(color=colors[variant], width=2),
                    hovertemplate=(
                        f"{variant} | {width} neurons | {training_step:,} training steps"
                        "<br>Grid MSE: %{y:.6g}<br>Seed: %{customdata[0]}"
                        "<br>Train MSE: %{customdata[1]:.6g}<extra></extra>"
                    ),
                ),
                row=panel,
                col=1,
            )
            trace_steps.append(training_step)
        means = [
            row
            for row in summary
            if row["training_steps"] == training_step and row["hidden_size"] == width
        ]
        if not means:
            continue
        figure.add_trace(
            go.Scatter(
                x=[row["variant"] for row in means],
                y=[max(row["mean"], plot_floor) for row in means],
                text=[f"{row['mean']:.3g}" for row in means],
                customdata=[[row["runs"]] for row in means],
                mode="markers+text",
                textposition="top center",
                textfont=dict(size=11),
                cliponaxis=False,
                visible=training_step == selected_step,
                marker=dict(
                    symbol="diamond",
                    color="#f59e0b",
                    size=13,
                    line=dict(color="#78350f", width=2),
                ),
                name="Arithmetic mean",
                legendgroup="mean",
                showlegend=show_mean_legend,
                hovertemplate=(
                    f"%{{x}} | {width} neurons | {training_step:,} training steps"
                    "<br>Mean MSE: %{y:.6g}<br>Runs: %{customdata[0]}<extra></extra>"
                ),
            ),
            row=panel,
            col=1,
        )
        trace_steps.append(training_step)
        show_mean_legend = False
for panel in range(1, len(report_widths) + 1):
    figure.update_yaxes(
        title_text="Grid MSE", type="log", showticklabels=True, row=panel, col=1
    )
figure.update_xaxes(
    title_text="Model / Euler evaluation steps",
    categoryorder="array",
    categoryarray=variants,
    tickmode="array",
    tickvals=variants,
    ticktext=["Standard<br>MLP"]
    + [f"CFM<br>{step}" for step in cfm_eval_sampling_steps],
    tickangle=0,
)
height = 480 * len(report_widths) + 120
figure.update_layout(
    title=dict(
        text=f"Grid MSE at {selected_step:,} training steps",
        x=0,
        xanchor="left",
        y=0.99,
        yanchor="top",
        font=dict(size=20),
    ),
    template="plotly_white",
    autosize=True,
    height=height,
    margin=dict(l=80, r=35, t=140, b=140),
    legend=dict(
        orientation="h", x=0.5, xanchor="center", y=-100 / (height - 280), yanchor="top"
    ),
    updatemenus=[
        dict(
            type="dropdown",
            direction="down",
            x=1,
            xanchor="right",
            y=1.02,
            yanchor="bottom",
            active=report_steps.index(selected_step),
            buttons=[
                dict(
                    label=f"{step:,} training steps",
                    method="update",
                    args=[
                        {"visible": [trace_step == step for trace_step in trace_steps]},
                        {"title.text": f"Grid MSE at {step:,} training steps"},
                    ],
                )
                for step in report_steps
            ],
        )
    ],
)
figure.show(config={"responsive": True, "displayModeBar": False})